# MODULE 46 — FACIAL RECOGNITION

## 46.4 Face Identification (1:N)

**Must Know**

*Computer Vision · Biometrics*

In the previous section, we studied Face Verification (1:1).

Now let's move to Face Identification.

The central question is:

> "Which enrolled identity does this face correspond to?"

Face identification is a biometric recognition process in which the system compares an unknown face against a database of enrolled identities to find a matching identity.

### Real-world example

Imagine an organization has 10,000 registered employees.

A person enters the building, and a camera captures their face.

The system must determine:

- Is this person enrolled?
- If enrolled, which employee are they?
- If no sufficiently good match exists, should the system return Unknown?

Unlike verification, the system doesn't know whose identity it should check beforehand.

---

## 1. Face identification pipeline

![Face identification pipeline](https://images.openai.com/static-rsc-4/38fg0I5UckNvCz0uhwVSPbhrcBtfCLVC12iOxptgb2FUJRnkgAhMDSAMrFpKY1w_mLesSE8FWJ-3plBimh_xn8SpnB9IaqbG4y2NtzIE8A5fFo7yC2dQDOL22ItLEnOST6qTFxK32tVz2BT0vyf7YEfWPOkuxhnOJ3j34ESLksg?purpose=inline)

[Original image source — DigitalSense](https://www.digitalsense.ai/blog/how-does-facial-recognition-work?utm_source=chatgpt.com)

```text
           Enrolled Face Database
                    |
                    ↓
            Stored Face Embeddings
                    |
                    |
Captured Image      |
       ↓            |
Face Detection      |
       ↓            |
Face Alignment      |
       ↓            |
Face Recognition Model
       ↓            |
Query Embedding     |
       |            |
       └──────┬─────┘
              ↓
        Similarity Search
              ↓
        Rank Candidates
              ↓
         Threshold Check
              ↓
        Identity / Unknown
```

There are two important components:

- **Gallery:** The database of enrolled face embeddings.
- **Probe:** The newly captured face that needs to be identified.

---

## 2. Understanding the Gallery and Probe

These are two important terms in biometric identification.

### A. Gallery

The gallery contains previously enrolled identities and their face embeddings.

For example:

| Employee ID | Identity | Stored embedding |
|---|---|---|
| E001 | Person A | Vector A |
| E002 | Person B | Vector B |
| E003 | Person C | Vector C |
| E004 | Person D | Vector D |

The database stores numerical representations rather than needing to compare every new photograph directly against every stored photograph.

### B. Probe

A probe is the newly captured face that the system wants to identify.

```text
New Face Image
      ↓
Face Recognition Model
      ↓
Query Embedding
```

The query embedding is compared with the embeddings stored in the gallery.

![Metric learning and image retrieval illustration](https://images.openai.com/static-rsc-4/z0wgrV_6_Y4VlnglW69V5t8hpgHPvV7kLp-0yYUueP6-SGENpJ5CZSGnmY_vM-Z2COJltDrG94muRlMSA-XZhpzlyDiHmw8I1JTUf63_PfShODlHG5ThhEivXxhDfpVtT-77_hA0EKMthBaZN7pmY5-eUeuo6TBjypJ8xey1VHQ?purpose=inline)

[Original image source — Metric Learning](https://kmhana.tistory.com/14?utm_source=chatgpt.com)

---

## 3. Similarity search

This is the core operation in face identification.

Suppose the system has four enrolled identities:

```text
Gallery:

Person A → Embedding A
Person B → Embedding B
Person C → Embedding C
Person D → Embedding D

New captured face → Query Embedding
```

The system calculates similarity between the query embedding and every enrolled embedding.

For example:

| Identity | Cosine similarity |
|---|---:|
| Person A | 0.42 |
| Person B | 0.91 |
| Person C | 0.67 |
| Person D | 0.35 |

The highest similarity belongs to Person B.

Therefore, Person B becomes the top candidate.

![Face similarity search illustration](https://images.openai.com/static-rsc-4/eT4lwB6O_X8ICoLluS0SZUl9hURVpTWbWA-56m8Muw5wgDdAjkHlYgS_5mYHcRthd3x4XfXqUQfZp3uUW6l8fW2aspoWTjlqdZd52pm0Ewpe4weoSCXwU5gRfTVvxXWPC1TCkJc3npmZ6lP3jba45U4CrGTgEx3EQnKXY1CvCGM?purpose=inline)

[Original image source — Tencent Cloud](https://cloud.tencent.com/developer/article/1005303?utm_source=chatgpt.com)

**Important:** The highest similarity does not automatically mean a valid identity match. We must also apply a matching threshold.

---

## 4. Ranking candidates

Identification systems can rank candidates according to their similarity scores.

This is called **Top-K retrieval**.

Suppose the similarity results are:

| Rank | Identity | Similarity |
|---:|---|---:|
| 1 | Person B | 0.91 |
| 2 | Person C | 0.67 |
| 3 | Person A | 0.42 |
| 4 | Person D | 0.35 |

Here:

- Top-1 = Person B
- Top-2 = Person B and Person C
- Top-3 = Person B, Person C and Person A

Top-K retrieval is useful when a system needs to provide several candidate identities for further checking.

---

## 5. Threshold-based identification

Now consider an important situation.

What happens if the captured face does not belong to anyone in the database?

The system must not simply assign the closest identity.

For example:

| Identity | Cosine similarity |
|---|---:|
| Person A | 0.31 |
| Person B | 0.45 |
| Person C | 0.38 |
| Person D | 0.29 |

Person B has the highest score, but the similarity may still be too low to accept.

Suppose the illustrative threshold is:

$$
T=0.80
$$

The highest score is:

$$
S_{\max}=0.45
$$

Since:

$$
0.45<0.80
$$

**UNKNOWN**

The system rejects the candidate rather than incorrectly assigning an identity.

### Identification decision rule

For cosine similarity:

$$
\boxed{
\text{Identity}=
\begin{cases}
\text{Top candidate}, & S_{\max}\geq T\\
\text{Unknown}, & S_{\max}<T
\end{cases}
}
$$

For Euclidean distance, the corresponding rule uses the minimum distance:

$$
\boxed{
\text{Identity}=
\begin{cases}
\text{Nearest candidate}, & d_{\min}\leq T\\
\text{Unknown}, & d_{\min}>T
\end{cases}
}
$$

The thresholds must be validated for the actual model and database.

---

## 6. Practical implementation using PyTorch

Let's simulate a simple face identification system.

We'll create:

- One query embedding.
- Four enrolled face embeddings.
- A similarity search.
- A threshold-based identity decision.

### Step 1 — Create the enrolled face database

```python
import torch
import torch.nn.functional as F

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

gallery = torch.tensor([
    [0.1, 0.2, 0.3, 0.4],
    [0.8, 0.7, 0.6, 0.5],
    [0.2, 0.9, 0.1, 0.3],
    [0.5, 0.2, 0.8, 0.1]
], device=device)

identities = [
    "Person A",
    "Person B",
    "Person C",
    "Person D"
]
```

Each row represents one enrolled identity's embedding.

### Step 2 — Create the query embedding

```python
query = torch.tensor(
    [[0.75, 0.65, 0.55, 0.45]],
    device=device
)
```

This represents the newly captured face after passing through the recognition model.

### Step 3 — Calculate cosine similarity

```python
gallery_normalized = F.normalize(
    gallery,
    p=2,
    dim=1
)

query_normalized = F.normalize(
    query,
    p=2,
    dim=1
)

similarities = torch.mm(
    query_normalized,
    gallery_normalized.T
)

print(similarities)
```

Here, `torch.mm()` calculates the similarity between the query and every enrolled embedding.

### Step 4 — Find the best candidate

```python
best_score, best_index = torch.max(
    similarities,
    dim=1
)

threshold = 0.80

if best_score.item() >= threshold:
    print("Identified:", identities[best_index.item()])
else:
    print("Identity: UNKNOWN")

print("Similarity:", best_score.item())
```

### What did we implement?

```text
Enrolled Face Embeddings
         +
      Query Embedding
          ↓
    Cosine Similarity
          ↓
     Candidate Ranking
          ↓
     Threshold Check
          ↓
     Identity / Unknown
```

This is a simulated identification system using manually created vectors. A real implementation needs a pretrained face recognition model to generate embeddings from actual face images.

---

## 7. What happens when the database becomes large?

Imagine different identification systems:

| System | Number of enrolled identities |
|---|---:|
| Small office | 100 |
| College | 10,000 |
| Large organization | 100,000 |
| Large-scale system | Millions |

If we have one query embedding and 100,000 stored embeddings, comparing the query against every stored embedding is called an **exhaustive or brute-force search**.

This is straightforward but can become computationally expensive as the database grows.

### Approximate Nearest Neighbor (ANN)

For large embedding databases, systems can use **Approximate Nearest Neighbor search**.

ANN methods efficiently retrieve embeddings that are likely to be close to the query without necessarily comparing against every stored vector.

Examples of vector search technologies include:

- **FAISS** — Efficient similarity search for dense vectors.
- **ScaNN** — Scalable nearest-neighbor search.

![Vector indexes and approximate nearest neighbor search](https://images.openai.com/static-rsc-4/2cM-jD6D1HdfE-ACApvfsDk9x0LIWlqYyVlIDjZRCMy_Fbz5pC8dnthhiyzU2BseQoN6i_MeG4WCcMamOaTnA7w0j7OyjZ2R3yAUrdlniHOpAQl_h8DT5D1lSioemb_NpouHtoEPb0EJI_JiOsnUT-b72wPXNvPivPVNWO0-l5M?purpose=inline)

[Original image source — Medium](https://grahamjroy.medium.com/vector-indexes-ann-hnsw-ivf-pq-and-how-rag-actually-retrieves-920746ebe5d2?utm_source=chatgpt.com)

**Good to Know**

ANN is a search optimization, not a different face recognition model. Its retrieval accuracy and speed must be evaluated for the application.

---

## 8. Face verification vs identification

This is an important interview question.

| Feature | Verification | Identification |
|---|---|---|
| Question | Are these two faces the same person? | Who is this person? |
| Type | 1:1 | 1:N |
| Reference | One enrolled identity | Entire enrolled gallery |
| Operation | Compare two embeddings | Search and rank embeddings |
| Output | Match / No Match | Identity / Unknown |
| Example | Smartphone unlock | Employee database search |

There are also two identification scenarios:

- **Closed-set identification:** The system assumes the person belongs to the enrolled gallery and returns the closest candidate.
- **Open-set identification:** The person may not be enrolled, so the system must support an Unknown result.

Real-world identification systems often need open-set recognition.

---

## 9. Industry applications

### A. Airport identification

![Airport face identification](https://images.openai.com/static-rsc-4/WlP6m6s2mdWLLsrhcW7caHjlnFWPdhJXsM-hHS7g9tcTqRjcNtMJpYOzl_zKlXSzWbQplPJxmoH8FMzrsEtWkLGghAbeRXffrqeiGwv-wBin5-L49YrYqFgvj7E7kOe-w1LWkf1RFe5GVjas5dwazSOM0CzX_sedR1nY3epin0k?purpose=inline)

Compare a captured face against enrolled passenger identities.

### B. Employee identification

![Employee face identification system](https://images.openai.com/static-rsc-4/TpYoU-ataSwJUmDcqH3kIdUx9cHty9hfzBRJ_GsISulV_IDj0adWRopscOlN2xDQitNU5RP_IvKKL70WXAQqCph-EeUGEBT3z4iUnNHu0vQxvIzC79H4Vy5GY91dyChNyFpnhY6rxzLFSCNeD71vGQpoe4EmY7huHY1qRpnye9Y?purpose=inline)

Search for an employee within an authorized identity database.

### C. Forensic investigation

![Forensic facial recognition investigation](https://images.openai.com/static-rsc-4/5htLPNtsL6z7TZkXMFcG7Yr5qy8CLkXpHcwPoGMWDLdZxv0VCrLBDXbIR2bOi6QqSwi4Otej7FCkq3xU51NAnI3IMS7W3pGimyO2iVFgSc4O3_aJzuIc1xXv2UQKwoGdwEH9sWXAwtIJMzZPa5gBOMNaJkUmk3LVkmiJCWhrzK4Q?purpose=inline)

Generate candidate matches for human review. A similarity result alone is not proof of identity.

---

## 10. What you should remember

1. Face identification is a 1:N biometric search.
2. Gallery = enrolled face embeddings.
3. Probe = newly captured face embedding.
4. Similarity search compares the probe against the gallery.
5. Top-K retrieval ranks the most similar candidates.
6. Threshold checking prevents blindly assigning the closest identity.
7. Open-set identification supports an Unknown result.
8. ANN techniques help scale similarity searches to large databases.

---
